## Project brief: two human GWAS 

- try:
  1. pancreatic cancer and 
  2. Alzheimer's disease
  3. or other disease/grade - since genomic data are protected

### Goal
Set up two human genome-wide association studies (GWAS), one for pancreatic cancer and one for Alzheimer's disease (AD). Run every association and genotype-QC step in PLINK (1.9 and/or 2.0), following the GWASTutorial (https://cloufield.github.io/GWASTutorial).

### Fixed conventions
- Reference genome: GRCh38 (use the no-ALT analysis set, e.g. `GCA_000001405.15_GRCh38_no_alt_analysis_set`). If a dataset is only available on GRCh37/hg19, flag this and propose a liftover (CrossMap or `bcftools +liftover`) rather than mixing builds.
- Environment: uv with Python 3.12. Wrap every shell command in Python (`subprocess.run(..., check=True)`, with logging). Command-line tools that pip/uv cannot install (FastQC, fastp, BWA-MEM2, samtools, bcftools, GATK, PLINK) go into one separate conda/mamba env (bioconda) or are installed as pinned static binaries. Record every tool version.
- Hardware: local Linux machine, 20 cores, 63 GB RAM, no GPU in the sandbox. Size threads and memory to fit.

### Phase 1: Find data (search only, no downloads)
Note: a GWAS needs a cohort of cases and controls (usually hundreds to thousands of people), not a single patient. "One dataset" here means one cohort per disease.

For each disease, find candidate datasets and report them in a table with these columns: name/accession, repository, data type (genotyping array / WES / WGS FASTQ / VCF / summary statistics), number of cases and controls, ancestry, genome build, access level (open vs. controlled, e.g. dbGaP/EGA/ADNI/NIAGADS), approximate size, and link.

1. Pancreatic cancer. Prefer low-grade disease and state which definition each dataset uses: tumour grade G1 (well-differentiated PDAC), low-grade precursor lesions (PanIN, low-grade IPMN), or low-grade pancreatic neuroendocrine tumour (PanNET G1). If no grade-stratified data exist, fall back to all-grade PDAC and say so.
2. Alzheimer's disease. Note whether cases are clinical, autopsy-confirmed, or proxy (family history), and whether the dataset is late-onset or early-onset AD.

Rank sources in this order:
1. Open individual-level raw data (FASTQ/BAM/CRAM).
2. Open individual-level genotypes (VCF or PLINK files).
3. Controlled-access individual-level data. List the application steps; do not apply.
4. Open GWAS summary statistics (e.g. GWAS Catalog). These can support clumping, look-ups and comparison, but not a de novo PLINK association test.

Stop condition: if neither disease has openly downloadable individual-level data, report what you found (diseases, grades/subtypes, access level) and stop. Then propose options, such as applying for controlled access, a summary-statistics-only analysis, or a practice run on the tutorial's 1000 Genomes data with a simulated phenotype. Wait for my decision.

### Phase 2: Download plan (I run the downloads)

- Do not download any file larger than about 100 MB. Instead, give me the `wget`/`curl` (or `prefetch`/`fasterq-dump`, `aspera`, EGA client) commands, plus file sizes, MD5/SHA checksums, and the total disk space needed.
- Use this target layout: `<project_root>/data/{raw,reference,interim,results}`. I will download into it and grant you access to `<project_root>`.
- For controlled-access data I will handle credentials myself. Never ask me to paste keys or tokens into the chat.
- Also list the GRCh38 reference files needed (FASTA, .fai, .dict, BWA-MEM2 index, dbSNP, known-indel and HapMap/1000G resource VCFs for BQSR/VQSR).

### Phase 3: Code to write (after my approval of Phases 1–2)
Write modular, re-runnable Python scripts (per-sample, idempotent, logged), as follows:

1. Read QC on FASTQ files: FastQC per file, aggregated with MultiQC. Optionally add adapter/quality trimming with fastp, then run QC again. Outputs: the MultiQC HTML report plus a per-sample summary table (read count, %Q30, GC content, adapter content, duplication).
2. FASTQ to VCF on GRCh38:
   1. Align with BWA-MEM2 (with read groups).
   2. Sort and index with samtools, and mark duplicates with GATK MarkDuplicates.
   3. Run BQSR.
   4. Call per-sample gVCFs with GATK HaplotypeCaller.
   5. Joint-genotype with GenomicsDBImport and GenotypeGVCFs.
   6. Filter variants with VQSR, or hard filters for small cohorts.
   7. Normalise with `bcftools norm` (split multiallelics, left-align).
   8. Outputs: final bgzipped and tabix-indexed VCF, plus alignment QC (samtools flagstat/stats, mosdepth coverage) and variant QC (`bcftools stats`, Ti/Tv).
3. VCF to PLINK, with QC following the tutorial: convert (`plink2 --vcf`); sample and variant missingness; MAF and HWE filters; heterozygosity; sex check; relatedness (KING); LD pruning; PCA for population structure.
   Note: if the chosen dataset already contains genotypes (array or VCF), skip steps 1–2 and start here.

Out of scope until I confirm: imputation, the association test itself, and downstream steps (Manhattan/QQ plots, clumping, annotation).

### Working style
Wait for my approval at the end of each phase. State every assumption explicitly, and cite the accession for every dataset you propose.


The stop condition is met: I found no openly downloadable individual-level case/control data for either disease, so I've stopped at Phase 1. Every cohort that could support a PLINK association test is controlled-access, and none of them is stratified by tumour grade.

## Pancreatic cancer
- **PanScan (dbGaP phs000206)** is a GWAS of PDAC susceptibility. The project's genotype and phenotype data cover 9,437 individuals (PanScan I and II cases and controls, plus PanScan III cases only). Access is through a dbGaP request, with a Data Use Certification (DUC) Agreement.
- **PanC4 (phs000648)** brings together over 8,000 participants from 9 case-control studies in the United States, Europe and Australia. It is also on dbGaP and also controlled.
- **Grade:** these are the largest European-ancestry pancreatic cancer GWAS, and both cover all-grade PDAC.
- **Closest match to "low grade": IPMN germline GWAS.**
  - One is the first GWAS of IPMN progression, with a polygenic hazard score, in 338 patients.
  - The other used the Mass General Brigham Biobank: of 68,931 individuals, 2,525 had IPMNs.
  - Neither cohort is openly deposited.
- **PanNET:** the only germline studies I found are a 2016 neuroendocrine tumour GWAS that was mostly small-intestine tumours, and a candidate-SNP study of 320 PanNET cases and 4,436 controls. Neither provides grade-stratified PanNET data.

## Alzheimer's disease (late-onset)
- **NIAGADS** holds the main AD cohorts. Individual-level GWAS genotypes and ADSP whole genomes require a Data Access Request under the NIH Genomic Data Sharing Policy. Examples:
  - ADGC sets such as NG00022 (ADC1, 2,768 samples) and NG00023 (ADC2, 925 samples), both genotyping arrays.
  - ADSP Release 4 WGS: 36,361 participants, 45% of non-European ancestry.
- **ADNI** is also controlled. It contains 819 subjects: 229 cognitively normal, 398 with MCI and 192 with AD.
- **Open data are summary statistics only.** NIAGADS Open Access provides knowledge bases and downloadable annotation data, and the GWAS Catalog has large AD meta-analyses. These cannot be used for a new PLINK association test.

The survey table also lists build and phenotype for each source. Some accessions in it came from prior knowledge rather than this search, such as the Kunkle 2019 AD summary statistics (NG00075) and Bellenguez 2022 (GCST90027158). They are marked "(verify)" and need confirming before any download.

## Options
- **A. Apply for controlled access:** dbGaP for PanScan and PanC4; NIAGADS for ADSP and ADGC; the ADNI application for ADNI. Expect weeks to months.
- **B. Practice run now:** the GWASTutorial 1000 Genomes data with a simulated phenotype, through the full PLINK QC and association workflow. For FASTQ → VCF practice, add a few open 1000 Genomes high-coverage samples (GRCh38).
- **C. Summary-statistics track:** open AD and pancreatic summary statistics, with PLINK clumping against a 1000 Genomes LD reference, look-ups and polygenic risk scores.

B and C can run while A is pending.

## Gaps in your conda environment (genome_env)
It has PLINK 1.9 only. The brief also needs `plink2` (for `--vcf` conversion and KING relatedness), `multiqc` and `mosdepth`, so these should be added before Phase 3.

Which option or options do you want to pursue?

[phase1_dataset_survey.md]({{artifact:002195b5-e5ef-43f4-9881-ea9a3324ebd6}}) has the full survey table and the options.

# Phase 1: Dataset survey (search only, nothing downloaded)

Date: 2026-10-02. Build is GRCh37/hg19 unless stated otherwise. Items marked "(verify)" are from prior knowledge and were not confirmed by a search in this session.

## Pancreatic cancer

| Accession | Repository | Data type | Cases / controls | Ancestry | Build | Access | Grade / subtype |
|---|---|---|---|---|---|---|---|
| phs000206 (PanScan I–III) | dbGaP | Illumina SNP arrays (HumanHap550, 610-Quad, OmniExpress) | PanScan I+II cases and controls + PanScan III cases, 9,437 subjects | European | hg19 | Controlled (dbGaP DAR) | All-grade PDAC, not stratified by grade |
| phs000648 (PanC4) | dbGaP | Illumina OmniExpressExome array | >8,000 participants, 9 case-control studies | European (US, Europe, Australia) | hg19 | Controlled (dbGaP DAR) | All-grade PDAC |
| PanScan I–III summary statistics (Wolpin 2014; Klein 2018) | dbGaP (inside phs000206) | Summary statistics | 5,117 / 8,845 (PanScan I–III) | European | hg19 | Controlled within dbGaP (verify whether open) | All-grade PDAC |
| IPMN progression GWAS (PMID 39639588) | Not deposited openly (verify) | Array | 338 IPMN patients (Cox model) | Not checked | Not checked | Not public | Includes low-grade IPMN (the closest match to "low grade") |
| IPMN GWAS, Mass General Brigham (MGB) Biobank (PMID 41060111) | MGB Biobank | Array | 2,525 IPMN / ~66,400 non-IPMN | Mostly European | Not checked | Institutional only | IPMN of any grade |
| NET GWAS (Endocr Relat Cancer 2016) | Not deposited openly (verify) | Affymetrix 6.0 | Mostly small-intestine NET; few pancreatic NET (PanNET) cases | European | hg19 | Not public | No grade-stratified PanNET data |
| FinnGen pancreatic cancer endpoint (verify) | FinnGen | Summary statistics | Several hundred to ~1,000 cases (verify) | Finnish | GRCh38 | Open download after registration (verify) | All-grade |

## Alzheimer's disease

| Accession | Repository | Data type | Cases / controls | Ancestry | Build | Access | Phenotype |
|---|---|---|---|---|---|---|---|
| ADNI | LONI IDA / NIAGADS | Illumina arrays + WGS | 819 subjects (229 cognitively normal, 398 MCI, 192 AD) in the ADNI-1 genotyped set | Mostly European | hg19 (arrays), GRCh38 (WGS) | Controlled (ADNI application) | Clinical diagnosis, late-onset AD (LOAD) |
| NG00022 / NG00023 (ADGC ADC1 / ADC2) | NIAGADS DSS | SNP arrays | 2,768 / 925 samples | Mostly European | hg19 | Controlled (NIAGADS DAR) | Clinical LOAD, part autopsy-confirmed |
| ADSP Release 4 WGS | NIAGADS DSS | WGS (CRAM / pVCF) | 36,361 participants, 45% non-European | Multi-ancestry | GRCh38 | Controlled (NIAGADS DAR) | Clinical + autopsy, LOAD |
| NG00075 (Kunkle 2019, IGAP) (verify) | NIAGADS Open Access | Summary statistics | 21,982 / 41,944 (stage 1) | European | hg19 | Open | Clinical + autopsy LOAD |
| GCST90027158 (Bellenguez 2022) (verify) | EBI GWAS Catalog | Summary statistics | ~111k cases (incl. proxy) / ~677k controls | European | GRCh38 | Open | Clinical + proxy (family history) |

## Result for the stop condition

The stop condition is met. No openly downloadable individual-level data (FASTQ/BAM/CRAM, VCF or PLINK) with case/control labels were found for either disease. Every cohort suitable for a PLINK association test is controlled-access (dbGaP or NIAGADS DAR, or an ADNI application). No grade-stratified pancreatic cancer GWAS cohort exists in public repositories. The closest match to "low grade" is IPMN germline GWAS, but those cohorts are small and not deposited openly. Open data exist only as summary statistics, which cannot be used for a de novo PLINK association test.

## Options for the decision
- A. Apply for controlled access: PanScan and PanC4 via a dbGaP DAR (eRA Commons account, institutional signing official, Data Use Certification, IRB statement), and ADNI and/or ADSP via a NIAGADS DSS DAR. Typical turnaround is weeks to months.
- B. Practice pipeline now: run the GWASTutorial 1000 Genomes data (open, about 500 samples) with a simulated case/control phenotype, through the full PLINK QC and association workflow. For FASTQ → VCF practice, use a few open 1000 Genomes high-coverage samples (ENA/SRA), lifted over or already on GRCh38.
- C. Summary-statistics track: download the open AD statistics (Bellenguez 2022 GCST90027158 / Kunkle NG00075) and an open pancreatic set (FinnGen). Then do PLINK `--clump` with the 1000 Genomes LD reference, look-ups, cross-disease comparison, and a polygenic risk score (PRS) applied to the 1000 Genomes samples.
- B and C can run while A is pending.


### Disk space

Sizes below are from the GitHub API and the tutorial's own file listings.

| Item | Size |
|---|---|
| Repository clone (`Cloufield/GWASTutorial`) | ~458 MB (468,559 KB reported by GitHub) |
| → of which the sample dataset zip `1KG.EAS.auto...missing.zip` | 63.8 MB (63,756,897 bytes), shipped inside the repo |
| Unzipped PLINK binary set | ~190 MB: 149M .bed, 40M .bim, 13K .fam |
| Tutorial intermediate outputs (QC-cleaned set, PCA, GRM, LD matrices, association results) | ~300 MB |
| **Core PLINK track total** | **~1 GB** |

So about **1 GB**, and nothing needs downloading separately: the sample dataset zip is included in `01_Dataset` when you clone the repository, so there is no need to download it again. You have ~46 GB free, which is ample.

Two caveats on the optional modules:
- The visualization module uses gwaslab, whose reference panel is larger: the processed 1000 Genomes EAS dataset (hg19) is about 2.8 GB. Its own sample summary statistics are small (~40 MB).
- Later modules (LD score regression, functional annotation with VEP/ANNOVAR, PRS) need extra reference databases I have not sized. Those can add several GB each; I'd check them only if you want to run those modules.

### Which diseases it covers

**None — the phenotype is simulated, not a real disease.** The genotypes are 504 East Asian individuals from the 1000 Genomes Project, a healthy population reference with no clinical labels, carrying 1,235,116 variants. Phenotypes were simulated with GCTA from the 1KG EAS genotypes, using `--simu-cc 250 254 --simu-hsq 0.8 --simu-k 0.5`: 250 cases and 254 controls, 80% heritability, 50% prevalence, with causal SNPs picked from a supplied `causal.snplist`. In the phenotype file, `B1` is the phenotype name, where 1 means control and 2 means case.

A 50% prevalence and 0.8 heritability are deliberately unrealistic — they make the signal easy to detect for teaching. The trait is labelled "B1", not any named condition.

The only real disease data in the tutorial is summary statistics: the gwaslab module uses processed type 2 diabetes summary statistics from Biobank Japan, restricted to HapMap3 variants on chromosome 7.

### Two consequences for our project

1. **Build mismatch.** The tutorial data is GRCh37, listed among its own key terms as "1000 Genomes Project (1KG), EAS, GRCh37". Our brief fixes GRCh38. For a practice run this does not matter, as long as we keep the tracks separate and never mix the two builds.
2. **No FASTQ files.** The tutorial starts from PLINK binaries, so Phase 3 steps 1 and 2 (FASTQ quality metrics and FASTQ → VCF) cannot be practiced on it. For those we need separate open data, for example a 1000 Genomes high-coverage sample from ENA. Whole-genome FASTQ runs 40–120 GB per sample, so I would restrict it to one chromosome or a downsampled subset to keep it within your free space.

If you want option B, tell me whether to include a small FASTQ exercise alongside the PLINK track, and I'll size the download before you fetch anything.

### Is this study available? GWA data on 1401 individuals from the PennCATH study of coronary artery disease (CAD)

#### query

1. **PennCATH GWAS** tutorial data 1401 individuals coronary artery disease download
- A guide to genome‐wide association analysis and post‐analytic interrogation - Reed - 2015 - Statistics in Medicine - Wil

https://onlinelibrary.wiley.com/doi/10.1002/sim.6605

- this tutorial, we will be using EIR-auto-GP, to train deep learning models for predicting coronary artery disease (CAD) risk based on genotype data and a mix of categorical and continuous tabular inputs (i.e., clinical data).

https://eir-auto-gp.readthedocs.io/en/latest/tutorials/a_using_eir_auto_gp/01_basic_tutorial.html

2. Efficient testing and effect size estimation for set-based genetic association inference via semiparametric multilevel m
https://arxiv.org/pdf/2003.05611

3. PGS Catalog - PGS000337 / Coronary artery disease (Polygenic Score)
https://www.pgscatalog.org/score/PGS000337/

4. Design of the Coronary ARtery DIsease Genome-Wide Replication And Meta-Analysis (CARDIoGRAM) Study
https://www.ahajournals.org/doi/abs/10.1161/circgenetics.109.899443

5. Genetic susceptibility variants associated with cardiovascular disease
https://image-ppubs.uspto.gov/dirsearch-public/print/downloadPdf/9617597

6. Large-scale genome-wide association study of coronary artery disease in genetically diverse populations - PMC
https://pmc.ncbi.nlm.nih.gov/articles/PMC9419655/

7. Applying Supervised Learning Algorithms and a New Feature Selection Method to Predict Coronary Artery Disease
https://arxiv.org/pdf/1402.0459

8. plmmr: an R package to fit penalized linear mixed models for genome-wide association data with complex correlation struc
https://arxiv.org/pdf/2502.01577

9. Interpreting coronary artery disease GWAS results: A functional genomics approach assessing biological significance
https://journals.plos.org/plosone/article?id=10.1371%2Fjournal.pone.0244904

In [1]:
import requests
cands = [
 "https://s3.amazonaws.com/pbreheny-data-sets/penncath.zip",
 "https://pbreheny.github.io/adv-gwas-tutorial/",
 "https://pbreheny.github.io/adv-gwas-tutorial/data.html",
 "http://www.stat-gen.org/book.e1/data/",
 "https://www.stat-gen.org/",
 "https://raw.githubusercontent.com/pbreheny/adv-gwas-tutorial/master/_quarto.yml",
]

for u in cands:
    try:
        r = requests.head(u, timeout=15, allow_redirects=True)
        print(r.status_code, r.headers.get("content-length"), r.headers.get("content-type"), u)
    except Exception as e:
        print("ERR", type(e).__name__, str(e)[:90], u)

404 None application/xml https://s3.amazonaws.com/pbreheny-data-sets/penncath.zip
200 1792373 text/html; charset=utf-8 https://pbreheny.github.io/adv-gwas-tutorial/
404 5254 text/html; charset=utf-8 https://pbreheny.github.io/adv-gwas-tutorial/data.html
404 None text/html; charset=utf-8 http://www.stat-gen.org/book.e1/data/
200 None text/html; charset=utf-8 https://www.stat-gen.org/
404 14 text/plain; charset=utf-8 https://raw.githubusercontent.com/pbreheny/adv-gwas-tutorial/master/_quarto.yml


### query: Reed 2015 "guide to genome-wide association analysis" PennCATH GWAStutorial.bed clinical.csv

1. A guide to genome‐wide association analysis and post‐analytic interrogation - Reed - 2015 - Statistics in Medicine - Wil
https://onlinelibrary.wiley.com/doi/10.1002/sim.6605

2. A guide to genome-wide association analysis and post-analytic interrogation
https://www.researchgate.net/publication/281588338_A_guide_to_genome-wide_association_analysis_and_post-analytic_interrogation

3. A tutorial on conducting genome‐wide association studies: Quality control and statistical analysis - PMC
https://pmc.ncbi.nlm.nih.gov/articles/PMC6001694/

4. GitHub - monogenea/GWAStutorial: Genome-wide association studies tutorial 🧬
https://github.com/monogenea/GWAStutorial

5. A guide to genome-wide association analysis and post-analytic interrogation - PubMed
https://pubmed.ncbi.nlm.nih.gov/26343929/

6. Multimarker genetic association tests for panel count data
https://arxiv.org/pdf/2609.03113

7. Efficient testing and effect size estimation for set-based genetic association inference via semiparametric multilevel m
https://arxiv.org/pdf/2003.05611

9. Google Scholar
https://scholar.google.com/scholar_lookup?doi=10.1002/sim.6605&amp=&pmid=26343929

10. GWAS tutorial: An Introduction (2015 - out of date)
https://pbreheny.github.io/adv-gwas-tutorial/

### Locating PennCATH


In [2]:
import requests, re, json, base64

api = "https://api.github.com/repos/pbreheny/adv-gwas-tutorial"

r = requests.get(api, timeout=20)

print(r.status_code, r.json().get("default_branch") if r.ok else r.text[:200])

if r.ok:
    br = r.json()["default_branch"]
    t = requests.get(f"{api}/git/trees/{br}?recursive=1", timeout=30).json()
    paths = [e["path"] for e in t.get("tree", []) if e["type"]=="blob"]
    print(len(paths)); print([p for p in paths if p.lower().endswith((".qmd",".rmd",".md",".yml",".r"))][:40])



200 master
22
['_navbar.yml', '_output.yml', 'imputation.rmd', 'index.rmd', 'joint_analysis.rmd', 'marginal_analysis.rmd', 'population_structure.rmd', 'post_gwas.rmd', 'quality_control.rmd', 'readme.md']


### Extracting data source link from tutorial text

~2024

https://github.com/pbreheny/adv-gwas-tutorial

In [3]:
raw = "https://raw.githubusercontent.com/pbreheny/adv-gwas-tutorial/master/"

hits = {}
for f in ["index.rmd","readme.md","quality_control.rmd"]:
    txt = requests.get(raw+f, timeout=20).text
    urls = sorted(set(re.findall(r"https?://[^\s\)\"'>\]]+", txt)))
    hits[f] = {"len": len(txt), "urls": urls[:20]}
    if f == "index.rmd":
        # keep the paragraphs mentioning the data
        for m in re.finditer(r"[^\n]*(?:penncath|PennCATH|1401|download|\.zip)[^\n]*", txt):
            print("IDX|", m.group(0)[:300])
            
print(json.dumps(hits, indent=1)[:2000])



{
 "index.rmd": {
  "len": 2967,
  "urls": [
   "https://dx.doi.org/10.1002/sim.6605",
   "https://onlinelibrary.wiley.com/doi/abs/10.1002/mpr.1608"
  ]
 },
 "readme.md": {
  "len": 935,
  "urls": [
   "https://github.com/pbreheny/adv-gwas-tutorial/issues",
   "https://github.com/pbreheny/plmm",
   "https://privefl.github.io/bigsnpr/",
   "https://privefl.github.io/bigstatsr/",
   "https://www.cog-genomics.org/plink/1.9/",
   "https://www.cog-genomics.org/plink/2.0/"
  ]
 },
 "quality_control.rmd": {
  "len": 22442,
  "urls": [
   "http://www.ncbi.nlm.nih.gov/pubmed/21239051",
   "https://academic.oup.com/gigascience/article/4/1/s13742-015-0047-8/2707533?login=false",
   "https://d1ypx1ckp5bo16.cloudfront.net/penncath/penncath.tar.gz",
   "https://d1ypx1ckp5bo16.cloudfront.net/penncath/penncath.zip",
   "https://en.wikipedia.org/wiki/Hardy-Weinberg_principle",
   "https://missing.csail.mit.edu/2020/course-shell/",
   "https://privefl.github.io/",
   "https://privefl.github.io/bigsnpr/"

### Reading data description

In [4]:
txt = requests.get(raw+"quality_control.rmd", timeout=20).text
print(txt[:3000])
for u in ["https://d1ypx1ckp5bo16.cloudfront.net/penncath/penncath.zip",
          "https://d1ypx1ckp5bo16.cloudfront.net/penncath/penncath.tar.gz"]:
    try:
        r = requests.head(u, timeout=20, allow_redirects=True)
        print("HEAD", r.status_code, r.headers.get("content-length"), u)
    except Exception as e:
        print("ERR", type(e).__name__, str(e)[:80], u)



pats = r"(hg1[89]|GRCh3[78]|build|NCBI3[67]|861|1401|SNPs|\.bim|\.fam|clinical|CAD|sex|age|tg|hdl|ldl)"
lines = txt.split("\n")
keep = [l for l in lines if re.search(pats, l) and not l.startswith(("#", "`", " ", "\t"))]
print("\n".join(keep[:70]))

---
title: 'GWAS tutorial: Data formats, summary statistics, and quality control'
author: Patrick Breheny and Tabitha Peter 
date: '`r format(Sys.Date(), "%B %d, %Y")`'
editor_options: 
  chunk_output_type: console
---

```{r knitr_setup, include=FALSE, purl=FALSE}
library(knitr)
library(kableExtra)
set.seed(1)
knitr::opts_knit$set(aliases=c(h = 'fig.height', w = 'fig.width'))
knitr::opts_chunk$set(comment="#", message=FALSE, collapse=TRUE, cache=FALSE, tidy=FALSE, fig.align="center")
knitr::knit_hooks$set(small.mar = function(before, options, envir) {
  if (before) par(mar = c(4, 4, .1, .1))
})
```

`R` packages we need to get started 

```{r}
library(data.table)
library(dplyr)
library(ggplot2)
```


Throughout, I'm going to use the `data.table` package to read in and work with data frames; feel free to use something else. With one exception, the files are just white space delimited text files, anything can open them.

# Getting the data

This tutorial will use data from the PennCATH 

### Extracting dataset details from tutorial source

In [10]:
pats = r"(hg1[89]|GRCh3[78]|build|NCBI3[67]|861|1401|SNPs|\.bim|\.fam|clinical|CAD|sex|age|tg|hdl|ldl)"
lines = txt.split("\n")
keep = [l for l in lines if re.search(pats, l) and not l.startswith(("#", "`", " ", "\t"))]
print("\n".join(keep[:70]))

knitr::opts_chunk$set(comment="#", message=FALSE, collapse=TRUE, cache=FALSE, tidy=FALSE, fig.align="center")
Throughout, I'm going to use the `data.table` package to read in and work with data frames; feel free to use something else. With one exception, the files are just white space delimited text files, anything can open them.
If PLINK installed correctly, you will see a message print out that looks like this: 
The data are given in "PLINK" format, which is the most common format for chip-based GWAS data (as of this writing). `PLINK` is an open-source whole genome association analysis toolset designed to perform a range of basic large-scale analyses in a computationally efficient manner. As part of the toolset, PLINK has its own file formats -- again, these have become the standard format in the GWAS field. Data stored in PLINK format come in 'bundles' -- that is, a PLINK data set consists of a set of files, not just one data file. It is worth knowing how to use the PLINK command li

### Requesting access to PennCATH

1. domain: d1ypx1ckp5bo16.cloudfront.net  
3. reason: Host of the PennCATH tutorial dataset (penncath.zip / penncath.tar.gz). I only need a HEAD request to report the exact file size and confirm the link is live before you download it — no file download.


In [5]:
for u in ["https://d1ypx1ckp5bo16.cloudfront.net/penncath/penncath.zip",
          "https://d1ypx1ckp5bo16.cloudfront.net/penncath/penncath.tar.gz"]:
    r = requests.head(u, timeout=30, allow_redirects=True)
    cl = r.headers.get("content-length")
    print(r.status_code, cl, round(int(cl)/1e6,1) if cl else None, "MB", r.headers.get("last-modified"), u.split("/")[-1])



200 142818657 142.8 MB Tue, 16 May 2023 16:24:16 GMT penncath.zip
200 142819504 142.8 MB Tue, 16 May 2023 16:24:16 GMT penncath.tar.gz


### Reading zip central dir via range request

In [6]:
import struct

u = "https://d1ypx1ckp5bo16.cloudfront.net/penncath/penncath.zip"

total = 142818657
tail = requests.get(u, headers={"Range": f"bytes={total-65536}-{total-1}"}, timeout=30).content
print(len(tail))

# locate End Of Central Directory
i = tail.rfind(b"PK\x05\x06")
ncd, cdsize, cdoff = struct.unpack("<HII", tail[i+10:i+20])
print("entries", ncd, "cd size", cdsize, "cd off", cdoff)
cd = requests.get(u, headers={"Range": f"bytes={cdoff}-{cdoff+cdsize-1}"}, timeout=30).content

p = 0; rows=[]
while p < len(cd) and cd[p:p+4] == b"PK\x01\x02":
    comp, uncomp = struct.unpack("<II", cd[p+20:p+28])
    nlen, elen, clen = struct.unpack("<HHH", cd[p+28:p+34])
    name = cd[p+46:p+46+nlen].decode("utf-8", "replace")
    rows.append((name, uncomp, comp))
    p += 46 + nlen + elen + clen

for n, un, co in sorted(rows, key=lambda x: -x[1]):
    print(f"{n:45s} uncompressed {un/1e6:9.2f} MB   in-zip {co/1e6:8.2f} MB")

print("TOTAL uncompressed MB", round(sum(r[1] for r in rows)/1e6, 1))


65536
entries 5 cd size 423 cd off 142818212
data/penncath.bed                             uncompressed    302.38 MB   in-zip   135.24 MB
data/penncath.bim                             uncompressed     24.05 MB   in-zip     7.57 MB
data/penncath.csv                             uncompressed      0.04 MB   in-zip     0.01 MB
data/penncath.fam                             uncompressed      0.03 MB   in-zip     0.00 MB
data/                                         uncompressed      0.00 MB   in-zip     0.00 MB
TOTAL uncompressed MB 326.5


### Inferring variant count and genome build

In [ ]:
import zlib, math

bed_bytes = [r[1] for r in rows if r[0].endswith(".bed")][0]
bytes_per_variant = math.ceil(1401/4)
nsnp = (bed_bytes - 3)/bytes_per_variant
print("bed bytes", bed_bytes, "| bytes/variant", bytes_per_variant, "| n variants =", nsnp)

# local-header offset for the .bim member, re-parsed from the central directory
p = 0; loc = {}
while p < len(cd) and cd[p:p+4] == b"PK\x01\x02":
    comp, uncomp = struct.unpack("<II", cd[p+20:p+28])
    nlen, elen, clen = struct.unpack("<HHH", cd[p+28:p+34])
    (lho,) = struct.unpack("<I", cd[p+42:p+46])
    loc[cd[p+46:p+46+nlen].decode()] = (lho, comp)
    p += 46 + nlen + elen + clen

lho, comp = loc["data/penncath.bim"]
hdr = requests.get(u, headers={"Range": f"bytes={lho}-{lho+200000}"}, timeout=30).content
nlen, elen = struct.unpack("<HH", hdr[26:30])
stream = hdr[30+nlen+elen:]
d = zlib.decompressobj(-15)
text = d.decompress(stream, 200000).decode("utf-8", "replace")
bim_head = text.split("\n")[:8]

print("\n".join(bim_head))

bed bytes 302377026 | bytes/variant 351 | n variants = 861473.0
1 rs10458597 0 564621 -9 C
1 rs12565286 0 721290 G C
1 rs12082473 0 740857 T C
1 rs3094315 0 752566 C T
1 rs2286139 0 761732 C T
1 rs11240776 0 765269 G A
1 rs2980319 0 777122 A T
1 rs2980300 0 785989 A G


### Verifying build agains Ensembl coordinates

A .bim file in a Genome-Wide Association Study (GWAS) is a text-based marker information file used alongside .bed and .fam files in the PLINK binary genotype format. [1, 2, 3] 
#### File Structure & Columns
A .bim file contains no header row and has six tab- or space-separated columns for every single genetic variant (typically single nucleotide polymorphisms, or SNPs): [3, 4] 

   1. Chromosome code: An integer (1-22, X, Y, or 0 for unplaced/unknown).
   2. Variant identifier: A standard name like an rsID (e.g., rs123456) or a custom ID assigned during pre-processing (e.g., chr1:1234567).
   3. Position in centimorgans (cM): Genetic map position (set to 0 if a genetic map is unavailable).
   4. Base-pair coordinate (bp): Physical base-pair coordinate on the chromosome (often based on genome builds like GRCh37/hg19 or GRCh38/hg38).
   5. Allele 1 (A1): Usually the minor allele or reference/alternative allele depending on the tool version and settings.
   6. Allele 2 (A2): Usually the major allele or the other reference allele. [3, 4, 5] 

#### Key Uses in GWAS

* Variant Metadata Reference: Provides the physical location and allele definitions for the binary genotypes stored in the companion .bed file. [2, 3] 
* Quality Control (QC): Filtered during initial pipeline steps using software like PLINK to remove poorly performing markers (e.g., based on minor allele frequency or missingness). [5, 6] 
* Positional Mapping: Used downstream to align summary statistics with reference panels or annotate variants with gene locations. [3] 

If you are working on a data processing pipeline, would you like help with:

* PLINK command line flags to manipulate or filter .bim files?
* Converting VCF files into .bed/.bim/.fam formats?
* Resolving allele mismatch errors between datasets?


[1] [https://www.cog-genomics.org](https://www.cog-genomics.org/plink/1.9/formats)
[2] [https://cloufield.github.io](https://cloufield.github.io/GWASTutorial/03_Data_formats/)
[3] [https://www.youtube.com](https://www.youtube.com/watch?v=lFlWyltF5-s&t=411)
[4] [https://github.com](https://github.com/AAlhendi1707/GWAS/blob/master/Data-pre-processing.md)
[5] [https://cloufield.github.io](https://cloufield.github.io/GWASTutorial/04_Data_QC/)
[6] [https://www.youtube.com](https://www.youtube.com/watch?v=762PVlyZJ-U)


In [8]:
def ens(host, rsid):
    try:
        r = requests.get(f"https://{host}/variation/human/{rsid}", params={"content-type":"application/json"}, timeout=25)
        if not r.ok: return f"HTTP {r.status_code}"
        return [m["location"] for m in r.json().get("mappings", []) if m.get("coord_system")=="chromosome"]
    except Exception as e:
        return f"ERR {type(e).__name__}"

for rsid, bimpos in [("rs3094315","1:752566"), ("rs12565286","1:721290"), ("rs2980300","1:785989")]:
    print(f"{rsid:12s} | bim:", bimpos, "| GRCh38:", ens("rest.ensembl.org", rsid), "| GRCh37:", ens("grch37.rest.ensembl.org", rsid))



rs3094315    | bim: 1:752566 | GRCh38: ['1:817186-817186'] | GRCh37: ['1:752566-752566']
rs12565286   | bim: 1:721290 | GRCh38: ['1:785910-785910'] | GRCh37: ['1:721290-721290']
rs2980300    | bim: 1:785989 | GRCh38: ['1:850609-850609'] | GRCh37: ['1:785989-785989']


### Tabulating chromosome compostion from .bim

In [13]:
import collections

blob = requests.get(u, headers={"Range": f"bytes={lho}-{lho+30+nlen+elen+comp-1}"}, timeout=120).content
raw = blob[30+nlen+elen:]
bim_txt = zlib.decompressobj(-15).decompress(raw).decode()
L = [l for l in bim_txt.split("\n") if l]

chrom = collections.Counter(l.split("\t")[0] if "\t" in l else l.split()[0] for l in L)
print("variants in .bim:", len(L))
print("chromosomes:", dict(sorted(chrom.items(), key=lambda kv: int(kv[0]))))

mono = sum(1 for l in L if l.split()[4] == "-9" or l.split()[5] == "-9")
print("variants with a '-9' (missing) allele code:", mono)

variants in .bim: 861473
chromosomes: {'1': 71038, '2': 73717, '3': 60565, '4': 55675, '5': 56178, '6': 54176, '7': 46391, '8': 48299, '9': 41110, '10': 47930, '11': 44213, '12': 42124, '13': 34262, '14': 28054, '15': 25900, '16': 27591, '17': 19939, '18': 26231, '19': 11482, '20': 22753, '21': 12463, '22': 11382}
variants with a '-9' (missing) allele code: 26154


### Reading phenotype table for case control counts

In [14]:
import io, pandas as pd
def member(name):
    lh, cm = loc[name]
    b = requests.get(u, headers={"Range": f"bytes={lh}-{lh+30+512+cm-1}"}, timeout=60).content
    nl, el = struct.unpack("<HH", b[26:30])
    return zlib.decompressobj(-15).decompress(b[30+nl+el:])

clin = pd.read_csv(io.BytesIO(member("data/penncath.csv")))
fam = pd.read_csv(io.BytesIO(member("data/penncath.fam")), sep=r"\s+", header=None,
                  names=["FID","IID","PAT","MAT","SEX","PHENO"])
print("clinical:", clin.shape, list(clin.columns))
print(clin["CAD"].value_counts(dropna=False).to_dict())
print("sex (1=male,2=female):", clin["sex"].value_counts().to_dict())
print("age:", clin["age"].describe()[["mean","min","max"]].round(1).to_dict())
print("missing per column:", clin.isna().sum().to_dict())
print("fam:", fam.shape, "| .fam PHENO values:", fam["PHENO"].value_counts().to_dict(), "| .fam SEX:", fam["SEX"].value_counts().to_dict())



clinical: (1401, 7) ['FamID', 'CAD', 'sex', 'age', 'tg', 'hdl', 'ldl']
{1: 933, 0: 468}
sex (1=male,2=female): {1: 937, 2: 464}
age: {'mean': 55.7, 'min': 22.0, 'max': 87.0}
missing per column: {'FamID': 0, 'CAD': 0, 'sex': 0, 'age': 0, 'tg': 92, 'hdl': 92, 'ldl': 119}
fam: (1401, 6) | .fam PHENO values: {1: 470, 0: 468, 2: 463} | .fam SEX: {1: 937, 2: 464}


### Cros tabulating fam phenotypes against CAD status

In [15]:
m = fam.merge(clin, left_on="FID", right_on="FamID", how="inner")
print("matched rows:", len(m))
print(pd.crosstab(m["PHENO"], m["CAD"]).to_string())
print("sex agreement .fam vs csv:", (m["SEX"]==m["sex"]).all())

matched rows: 1401
CAD      0    1
PHENO          
0      468    0
1        0  470
2        0  463
sex agreement .fam vs csv: True


**Yes — PennCATH is openly downloadable, with no application.** It is the only real-patient, individual-level GWAS dataset I've found that is genuinely open, so it meets rank 2 of your priority list. The catch is that the disease is CAD, not pancreatic cancer or Alzheimer's.

I verified everything below against the live archive using HTTP range requests, reading only the zip directory, the 7.6 MB `.bim` and the two tiny text files. I did not download the 302 MB `.bed`.

## What it is
| Field | Value |
|---|---|
| Study | PennCATH (Penn Catheterization study), CAD genetic risk factors — PMID 21239051 |
| Tutorial | Reed et al. 2015 (doi:10.1002/sim.6605); maintained version by Breheny & Peter |
| Access | Open, direct HTTP, no credentials |
| Archive | 142.8 MB (`penncath.tar.gz` or `.zip`, identical), last modified 2023-05-16 |
| Samples | **1,401 individuals** |
| Variants | **861,473** autosomal |
| Cases / controls | **933 CAD cases / 468 controls** |
| Sex | 937 male / 464 female |
| Age | mean 55.7, range 22–87 |
| Covariates | age, sex, triglycerides, HDL, LDL (92–119 missing values in the lipid columns) |
| Build | **GRCh37/hg19** |

The variant count is self-consistent: the `.bed` is 302,377,026 bytes, which is exactly 3 + 351 × 861,473, and the `.bim` has 861,473 lines.

The build is confirmed rather than assumed. rs3094315 sits at 1:752566 in the `.bim`, which is its GRCh37 position; in GRCh38 it is at 1:817186. Two further SNPs gave the same result.

## Disk space
| Item | Size |
|---|---|
| Downloaded archive | 143 MB |
| Unzipped: bed 302.38 MB + bim 24.05 MB + csv 38 KB + fam 29 KB | 327 MB |
| QC, PCA and association outputs | ~400 MB |
| **Subtotal** | **~900 MB** |
| Optional GRCh38 liftover (second copy + intermediate VCF) | ~700 MB |

Budget **1.6 GB** if you want the liftover, 900 MB without. Either is trivial against your 49 GB free.

```bash
cd <project_root>/data/raw
wget -c https://d1ypx1ckp5bo16.cloudfront.net/penncath/penncath.tar.gz
tar -xzf penncath.tar.gz     # creates data/penncath.{bed,bim,csv,fam}
```
No published checksum exists for this file, so I'd compute an `sha256sum` after download and record it for reproducibility.

## Four things to handle in QC
1. **The `.fam` phenotype column is miscoded and must not be used.** Controls are coded 0 (468 individuals) and cases are split across 1 (470) and 2 (463). Under PLINK's convention — 1 control, 2 case, 0 missing — that would invert the cases and discard the controls. The phenotype has to come from `penncath.csv` via `--pheno`. The 1-versus-2 split nests perfectly inside CAD=1, so it is probably an unlabelled case subtype.
2. **26,154 variants carry a `-9` missing allele code.** This is exactly what makes the tutorial's first `--maf` call fail; the files need rewriting with `--make-bed` first.
3. **No sex chromosomes** (chr1–22 only), so `--check-sex` is impossible. The only sex check available is `.fam` against the csv, and they agree for all 1,401 samples.
4. **GRCh37.** Your brief fixes GRCh38, so either lift over (convert to VCF, `CrossMap`/`Picard LiftoverVcf` with the hg19→hg38 chain, then back to PLINK, losing a small share of variants) or record GRCh37 as a declared exception for this dataset. For QC and association the build is irrelevant; it matters only when annotating hits or merging with other data.

## How this compares with the options
PennCATH is a better practice vehicle than the 1000 Genomes track in one important way: the phenotype is real. The tutorial's simulated trait has 80% heritability and 50% prevalence, so the association step is unrealistically easy. PennCATH gives real effect sizes, real confounding and a real case/control imbalance. A 933/468 study is small by modern standards and will likely recover only the 9p21 locus, which is the honest outcome for a cohort this size.

It does not, however, contain FASTQ files, so Phase 3 steps 1 and 2 still need separate data.

Three ways forward, and I'd suggest the first:
- **Run the full PLINK workflow on PennCATH** (real CAD data), and separately fetch one small open FASTQ sample for the quality-metrics and VCF-calling code.
- Keep PennCATH as practice and still apply for dbGaP/NIAGADS access for the two diseases you actually want.
- Drop CAD and go with the summary-statistics track for pancreatic cancer and AD.

Which do you want? If it's the first, tell me whether to include the liftover to GRCh38 or to stay on GRCh37.

[phase1_dataset_survey.md]({{artifact:bbc02b56-af16-4b64-a2d0-7a13d5891e83}}) — now version 2, with the full PennCATH fact sheet added.